# promptwall: transformer classifier (train + benchmark on a free GPU)

Fine-tunes **DeBERTa-v3-xsmall** as promptwall's ML layer and benchmarks it against the shipped TF-IDF model.

**Controlled comparison:** the training data comes from the same `train_classifier.build()` function as the TF-IDF model (same splits, same 3 training-only attack templates, same validation pages), and the threshold uses the same calibration rule. Only the model changes.

**Before running:** `Runtime > Change runtime type > T4 GPU`. Then `Runtime > Run all`. Total time is roughly 30-40 minutes, most of it the two benchmark runs.

In [ ]:
# 1. Settings - change REPO if your GitHub repo has a different name
REPO = "https://github.com/Abhigyanv23/promptwall.git"
MODEL_DIR = "models/promptwall-deberta-xsmall"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code and install it (transformer extra = torch, transformers, sentencepiece)
import os
if not os.path.exists("promptwall"):
    !git clone -q {REPO} promptwall
%cd /content/promptwall
!git pull -q
!pip install -q -e ".[dev,transformer]"


In [ ]:
# 3. Public datasets the benchmark is built from (~830 MB, a few minutes)
!bash bench/fetch_sources.sh

In [ ]:
# 4. Sanity checks: tests pass, and the shipped TF-IDF benchmark reproduces on Colab.
#    The diff below should show ONLY the latency line changing (hardware differs);
#    any changed detection or false-positive number means the setup differs.
!python -m pytest -q
!python bench/run_benchmark.py > /dev/null
!git diff -U0 -- bench/results/RESULTS.md


In [ ]:
# 5. Fine-tune (a few minutes on a T4). Watch: val AUC per epoch, the calibrated threshold,
#    val PAGE FPR, and the probes - especially the paraphrase the TF-IDF model misses (0.19).
!python bench/train_transformer.py --out {MODEL_DIR}

In [ ]:
# 6. Full benchmark with the transformer on every chunk (GPU).
#    Writes bench/results/RESULTS_transformer.md and friends; the TF-IDF results are untouched.
import os
assert os.path.exists(f"{MODEL_DIR}/promptwall_meta.json"), "no trained model - check step 5's output"
!python bench/run_benchmark.py --classifier transformer --model {MODEL_DIR} --device cuda

In [ ]:
# 7. Cascade: TF-IDF first, transformer only on chunks TF-IDF scores >= 0.02.
#    Much cheaper on CPU; this run measures how much recall that costs.
import os
assert os.path.exists(f"{MODEL_DIR}/promptwall_meta.json"), "no trained model - check step 5's output"
!python bench/run_benchmark.py --classifier transformer --model {MODEL_DIR} --device cuda --cascade

In [ ]:
# 8. CPU latency (what a laptop or server without a GPU would see), on 40 real pages.
import os
assert os.path.exists(f"{MODEL_DIR}/promptwall_meta.json"), "no trained model - check step 5's output"
import sys, time, statistics
sys.path[:0] = ["bench", "."]
import cases
from promptwall import Scanner
from promptwall.ml import InjectionClassifier
from promptwall.transformer_ml import TransformerClassifier

pages = [h for _, h in cases.benign_pages()][:40]
def per_page_ms(scanner):
    scanner.scan_html(pages[0])                      # warm-up
    ts = []
    for h in pages:
        t = time.perf_counter(); scanner.scan_html(h); ts.append((time.perf_counter() - t) * 1000)
    ts.sort(); return round(statistics.median(ts)), round(ts[int(0.95 * len(ts)) - 1])

tfidf = InjectionClassifier.load_default()
rows = {
    "TF-IDF (shipped)": Scanner(),
    "transformer, every chunk": Scanner(classifier=TransformerClassifier(MODEL_DIR, device="cpu")),
    "transformer, cascade": Scanner(classifier=TransformerClassifier(MODEL_DIR, device="cpu",
                                                                     prefilter=tfidf)),
}
print("CPU latency per page (p50 / p95 ms):")
for name, sc in rows.items():
    print(f"  {name:26s} {per_page_ms(sc)}")
c = rows["transformer, cascade"].classifier.stats
print(f"cascade sent {c['sent_to_transformer']} of {c['chunks']} chunks to the transformer")

In [ ]:
# 9. Download the results to commit (one file at a time - no zip)
from google.colab import files
for f in ["RESULTS_transformer.md", "results_transformer.json", "misses_transformer.csv",
          "per_case_transformer.jsonl", "RESULTS_transformer_cascade.md",
          "results_transformer_cascade.json", "misses_transformer_cascade.csv",
          "per_case_transformer_cascade.jsonl"]:
    files.download(f"bench/results/{f}")
files.download(f"{MODEL_DIR}/promptwall_meta.json")

## 10. Optional: publish the model on Hugging Face

The model (~280 MB) is too big for a GitHub repo, so it lives on the Hugging Face Hub and `TransformerClassifier("your-name/promptwall-deberta-xsmall")` downloads it.
This needs a token with **write** access (your existing `promptwall-laptop` token is read-only): create one at huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import create_repo, login, upload_folder
login()                                             # paste a WRITE token
HF_REPO = "YOUR-HF-USERNAME/promptwall-deberta-xsmall"   # <- change
create_repo(HF_REPO, exist_ok=True)
upload_folder(repo_id=HF_REPO, folder_path=MODEL_DIR)
print("uploaded:", f"https://huggingface.co/{HF_REPO}")